# 02 — Train Vanilla GISMo (baseline)

Trains the no-health-awareness baseline (vanilla GISMo). The checkpoint is the prerequisite for the `eval_filter_baseline.py` post-hoc filter and the reference for our MRR comparison.

**Runtime**: Colab free T4 GPU, ~30 min.

**Steps**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell 5) to wherever you put `data/` in your Drive
3. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# torch_geometric needs to be matched with the installed torch wheel.
# Colab's default torch is recent enough that the generic install works.
!pip install -q torch_geometric pandas numpy matplotlib

## 2. Get the code

In [ ]:
# Clone (or refresh) the repo
import os
if not os.path.exists('/content/CS471-Project'):
    !git clone https://github.com/lee1june61/CS471-Project.git /content/CS471-Project
else:
    !cd /content/CS471-Project && git pull --ff-only
%cd /content/CS471-Project

## 3. Mount Drive for data + outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === EDIT THIS if your Drive layout is different ===
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# sanity check
expected = ['flavorgraph_edges.csv', 'nodes_filtered.csv',
             'pairs_train.csv', 'pairs_val.csv', 'pairs_test.csv',
             'recipes.json', 'usda_mapping.json']
missing = [f for f in expected if not os.path.exists(f'{DATA_DIR}/{f}')]
if missing:
    raise FileNotFoundError(f'missing data files in {DATA_DIR}: {missing}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

## 4. Smoke test (optional)

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Skip by changing False below.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v1.py --mode baseline --max_epochs 1 --patience 1 \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v1_baseline
    print('\n[smoke] OK')

## 5. Train vanilla GISMo

In [ ]:
!python src/train_v1.py --mode baseline \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/baseline

print('\n[done] checkpoint:', f'{OUTPUT_DIR}/baseline/best_baseline.pt')

## 6. Quick check

In [ ]:
import json
with open(f'{OUTPUT_DIR}/baseline/test_predictions_baseline.json') as f:
    pred = json.load(f)
print('Vanilla GISMo test metrics:')
for k, v in pred['metrics'].items():
    print(f'  {k}: {v:.2f}')